# 📓 Notebook 01 — ดึงข้อมูล MT5 & คำนวณ SMC Indicators

## ในบทนี้คุณจะได้เรียนรู้:
- ดึงข้อมูลราคา XAUUSD จาก MetaTrader5 (หรือ CSV fallback)
- คำนวณ SMC indicators ด้วย `smartmoneyconcepts` library โดยตรง
- สร้าง candlestick chart พร้อม indicators ใน notebook

> ⚠️ **ไม่ต้องรัน Backend** — ทุกอย่างทำใน notebook นี้เลย!

## Data Flow
```
MetaTrader5 (หรือ CSV)
       │
       ▼
  pandas DataFrame (OHLCV)
       │
       ▼
  smartmoneyconcepts library
       │
  ┌────┼────┬────┬────┬────┐
  FVG  BOS  OB   Liq  Swing
```

In [ ]:
# ติดตั้ง dependencies
%pip install smartmoneyconcepts pandas numpy matplotlib mplfinance python-dotenv -q

# ลองติดตั้ง MetaTrader5 (ใช้ได้เฉพาะ Windows)
try:
    %pip install MetaTrader5 -q
    print('✅ MetaTrader5 ติดตั้งสำเร็จ')
except:
    print('⚠️ MetaTrader5 ไม่สามารถติดตั้งได้ (ไม่ใช่ Windows?) — จะใช้ CSV fallback')

print('✅ Dependencies ติดตั้งเสร็จ!')

## 📊 Cell 2: ดึงข้อมูลราคา XAUUSD

ระบบจะ **ลองเชื่อมต่อ MT5 ก่อน** ถ้าไม่ได้จะ fallback ไปใช้ CSV

In [ ]:
import pandas as pd
import numpy as np
import os

# ========== ฟังก์ชันดึงข้อมูล ==========

def get_mt5_data(symbol='XAUUSD', timeframe_str='15m', count=500):
    """
    ดึงข้อมูลจาก MetaTrader5 โดยตรง
    Returns: DataFrame หรือ None ถ้า MT5 ไม่พร้อม
    """
    try:
        import MetaTrader5 as mt5
        
        if not mt5.initialize():
            print(f'❌ MT5 initialize failed: {mt5.last_error()}')
            return None
        
        # แปลง timeframe string → MT5 constant
        tf_map = {
            '1m': mt5.TIMEFRAME_M1, '5m': mt5.TIMEFRAME_M5,
            '15m': mt5.TIMEFRAME_M15, '1h': mt5.TIMEFRAME_H1,
            '4h': mt5.TIMEFRAME_H4, '1d': mt5.TIMEFRAME_D1,
            '1w': mt5.TIMEFRAME_W1,
        }
        mt5_tf = tf_map.get(timeframe_str)
        if mt5_tf is None:
            print(f'❌ Timeframe ไม่รองรับ: {timeframe_str}')
            mt5.shutdown()
            return None
        
        rates = mt5.copy_rates_from_pos(symbol, mt5_tf, 0, count)
        mt5.shutdown()
        
        if rates is None or len(rates) == 0:
            print(f'❌ ไม่ได้รับข้อมูลจาก MT5')
            return None
        
        df = pd.DataFrame(rates)
        df['date'] = pd.to_datetime(df['time'], unit='s')
        df = df.rename(columns={'open':'open','high':'high','low':'low','close':'close','tick_volume':'volume'})
        df = df[['date','open','high','low','close','volume']].copy()
        
        print(f'✅ MT5: ได้ {len(df)} แท่งเทียน {timeframe_str} ({symbol})')
        return df
    
    except ImportError:
        print('⚠️ MetaTrader5 ไม่ได้ติดตั้ง')
        return None
    except Exception as e:
        print(f'❌ MT5 Error: {e}')
        return None


def generate_sample_csv(filepath='sample_xauusd.csv', count=500):
    """
    สร้าง CSV ตัวอย่างสำหรับทดสอบ (ถ้าไม่มี MT5 และไม่มี CSV)
    ใช้ random walk ที่ดูคล้ายราคาทองคำ
    """
    np.random.seed(42)
    base = 2650.0
    changes = np.random.randn(count) * 2.5
    prices = base + np.cumsum(changes)
    
    data = []
    for i, price in enumerate(prices):
        o = price + np.random.randn() * 1.5
        h = max(o, price) + abs(np.random.randn()) * 3
        l = min(o, price) - abs(np.random.randn()) * 3
        c = price
        v = int(abs(np.random.randn()) * 1000 + 500)
        dt = pd.Timestamp('2025-01-01') + pd.Timedelta(minutes=15*i)
        data.append({'date': dt, 'open': round(o,2), 'high': round(h,2), 
                     'low': round(l,2), 'close': round(c,2), 'volume': v})
    
    df = pd.DataFrame(data)
    df.to_csv(filepath, index=False)
    print(f'✅ สร้าง CSV ตัวอย่าง: {filepath} ({len(df)} แท่ง)')
    return df


def get_data(symbol='XAUUSD', timeframe='15m', count=500, csv_path='sample_xauusd.csv'):
    """
    ดึงข้อมูล: ลอง MT5 ก่อน → ถ้าไม่ได้ใช้ CSV → ถ้าไม่มี CSV สร้างตัวอย่าง
    """
    # 1. ลอง MT5
    df = get_mt5_data(symbol, timeframe, count)
    if df is not None:
        return df
    
    # 2. ลอง CSV
    if os.path.exists(csv_path):
        print(f'📁 Fallback: ใช้ CSV ไฟล์ {csv_path}')
        df = pd.read_csv(csv_path, parse_dates=['date'])
        df = df.tail(count).reset_index(drop=True)
        print(f'✅ CSV: ได้ {len(df)} แท่งเทียน')
        return df
    
    # 3. สร้าง CSV ตัวอย่าง
    print(f'📝 ไม่มี MT5 และไม่มี CSV → สร้างข้อมูลจำลอง...')
    return generate_sample_csv(csv_path, count)


# ========== ดึงข้อมูล ==========
df = get_data(timeframe='15m', count=200)
print(f'\n📊 ข้อมูล 5 แท่งล่าสุด:')
print(df.tail(5).to_string(index=False))

## 🔬 Cell 3: คำนวณ SMC Indicators ด้วย `smartmoneyconcepts`

**นี่คือ core ของระบบ** — เราจะเรียก library โดยตรง ไม่ต้องผ่าน backend!

SMC library ต้องการ DataFrame ที่มี columns: `open`, `high`, `low`, `close`

In [ ]:
import smartmoneyconcepts as smc

# ========== คำนวณ SMC Indicators ทุกตัว ==========

def compute_smc_indicators(df, swing_length=None):
    """
    คำนวณ SMC indicators ทุกตัว จาก DataFrame ที่มี OHLCV
    
    Returns:
        dict ของ DataFrames: swing, fvg, bos_choch, ob, liquidity, sessions, retracement
    """
    # ปรับ swing_length อัตโนมัติตามจำนวนข้อมูล
    if swing_length is None:
        if len(df) >= 200:
            swing_length = 20
        elif len(df) >= 100:
            swing_length = 15
        elif len(df) >= 50:
            swing_length = 10
        else:
            swing_length = max(5, len(df) // 6)
    
    print(f'📊 คำนวณ SMC indicators (swing_length={swing_length}, {len(df)} แท่ง)...')
    
    # 1. Swing Highs/Lows — พื้นฐานที่ indicators อื่นใช้
    swing = smc.smc.swing_highs_lows(df, swing_length=swing_length)
    
    # 2. Fair Value Gap
    fvg = smc.smc.fvg(df, join_consecutive=False)
    
    # 3. Break of Structure / Change of Character
    bos_choch = smc.smc.bos_choch(df, swing, close_break=True)
    
    # 4. Order Blocks
    ob = smc.smc.ob(df, swing, close_mitigation=False)
    
    # 5. Liquidity
    liquidity = smc.smc.liquidity(df, swing, range_percent=0.02)
    
    # 6. Sessions
    try:
        sessions = smc.smc.sessions(df, session='Custom', start_time='09:30', end_time='16:00', time_zone='UTC')
    except:
        sessions = pd.DataFrame()
    
    # 7. Retracements
    retracement = smc.smc.retracements(df, swing)
    
    print('✅ คำนวณ SMC indicators เสร็จ!')
    print(f'   Swing points: {swing["HighLow"].notna().sum()} จุด')
    print(f'   FVG: {(fvg["FVG"].notna() & (fvg["FVG"] != 0)).sum()} gaps')
    print(f'   BOS: {(bos_choch["BOS"].notna()).sum()} | CHOCH: {(bos_choch["CHOCH"].notna()).sum()}')
    print(f'   OB: {(ob["OB"].notna() & (ob["OB"] != 0)).sum()} blocks')
    print(f'   Liquidity: {(liquidity["Liquidity"].notna() & (liquidity["Liquidity"] != 0)).sum()} pools')
    
    return {
        'swing': swing,
        'fvg': fvg,
        'bos_choch': bos_choch,
        'ob': ob,
        'liquidity': liquidity,
        'sessions': sessions,
        'retracement': retracement,
    }


# คำนวณ!
indicators = compute_smc_indicators(df)

## 📈 Cell 4: สร้าง SMC Chart ด้วย Matplotlib

เราจะสร้างกราฟ candlestick พร้อม indicators ทั้งหมด — เหมือนที่ backend สร้างให้

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.style.use('dark_background')

def plot_smc_chart(df, indicators, last_n=50, figsize=(16, 8)):
    """
    สร้างกราฟ candlestick พร้อม SMC indicators
    """
    # ดึง N แท่งสุดท้ายเท่านั้น
    start = max(0, len(df) - last_n)
    window = df.iloc[start:].copy()
    swing = indicators['swing'].iloc[start:]
    fvg = indicators['fvg'].iloc[start:]
    bos_choch = indicators['bos_choch'].iloc[start:]
    ob = indicators['ob'].iloc[start:]
    liquidity = indicators['liquidity'].iloc[start:]
    
    fig, ax = plt.subplots(figsize=figsize, facecolor='#0C0E12')
    ax.set_facecolor('#0C0E12')
    
    n = len(window)
    opens = window['open'].values
    highs = window['high'].values
    lows  = window['low'].values
    closes = window['close'].values
    
    # --- Candlesticks ---
    for i in range(n):
        color = '#77dd76' if closes[i] >= opens[i] else '#ff6962'
        ax.plot([i, i], [lows[i], highs[i]], color=color, linewidth=1)
        h = abs(closes[i] - opens[i])
        b = min(opens[i], closes[i])
        ax.add_patch(Rectangle((i-0.3, b), 0.6, max(h, 0.1), facecolor=color, edgecolor=color, alpha=0.8))
    
    # --- FVG ---
    for i in range(len(fvg)):
        if pd.notna(fvg['FVG'].iloc[i]) and fvg['FVG'].iloc[i] != 0:
            top, bot = fvg['Top'].iloc[i], fvg['Bottom'].iloc[i]
            mit = fvg['MitigatedIndex'].iloc[i] if 'MitigatedIndex' in fvg.columns else None
            if pd.isna(mit):
                ax.add_patch(Rectangle((i, bot), n-i, top-bot, facecolor='yellow', alpha=0.2, edgecolor='yellow', lw=0.5))
    
    # --- Order Blocks ---
    for i in range(len(ob)):
        if pd.notna(ob['OB'].iloc[i]) and ob['OB'].iloc[i] != 0:
            top, bot = ob['Top'].iloc[i], ob['Bottom'].iloc[i]
            mit = ob['MitigatedIndex'].iloc[i] if 'MitigatedIndex' in ob.columns else None
            if pd.isna(mit):
                c = 'purple' if ob['OB'].iloc[i] == 1 else 'magenta'
                ax.add_patch(Rectangle((i, bot), n-i, top-bot, facecolor=c, alpha=0.25, edgecolor=c, lw=1))
    
    # --- BOS / CHOCH ---
    for i in range(len(bos_choch)):
        if pd.notna(bos_choch['BOS'].iloc[i]):
            level = bos_choch['Level'].iloc[i]
            broken = int(bos_choch['BrokenIndex'].iloc[i]) - start if pd.notna(bos_choch['BrokenIndex'].iloc[i]) else i
            c = 'lime' if bos_choch['BOS'].iloc[i] == 1 else 'red'
            if 0 <= broken < n:
                ax.plot([i, broken], [level, level], color=c, lw=2, alpha=0.9)
                lbl = 'BOS↑' if bos_choch['BOS'].iloc[i] == 1 else 'BOS↓'
                ax.text(broken, level, lbl, color=c, fontsize=8, ha='left',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        if pd.notna(bos_choch['CHOCH'].iloc[i]):
            level = bos_choch['Level'].iloc[i]
            broken = int(bos_choch['BrokenIndex'].iloc[i]) - start if pd.notna(bos_choch['BrokenIndex'].iloc[i]) else i
            if 0 <= broken < n:
                ax.plot([i, broken], [level, level], color='orange', lw=2, alpha=0.9, ls='--')
                lbl = 'CHoCH↑' if bos_choch['CHOCH'].iloc[i] == 1 else 'CHoCH↓'
                ax.text(broken, level, lbl, color='orange', fontsize=7, ha='left',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    
    # --- Swing Points ---
    for idx, row in swing.iterrows():
        if pd.notna(row.get('HighLow')):
            local_i = idx - swing.index[0]
            if row['HighLow'] == 1:
                ax.scatter([local_i], [row['Level']], color='red', marker='v', s=80, zorder=5)
            else:
                ax.scatter([local_i], [row['Level']], color='lime', marker='^', s=80, zorder=5)
    
    # --- Liquidity ---
    for i in range(len(liquidity)):
        if pd.notna(liquidity['Liquidity'].iloc[i]) and liquidity['Liquidity'].iloc[i] != 0:
            level = liquidity['Level'].iloc[i]
            swept = liquidity['Swept'].iloc[i] if 'Swept' in liquidity.columns else None
            c = 'gray' if pd.notna(swept) else 'gold'
            end = int(liquidity['End'].iloc[i]) - start if pd.notna(liquidity['End'].iloc[i]) else n-1
            if 0 <= end < n:
                ax.plot([i, end], [level, level], color=c, lw=2, alpha=0.8)
    
    ax.set_xlim(-0.5, n-0.5)
    ax.set_ylim(lows.min() * 0.999, highs.max() * 1.001)
    ax.tick_params(colors='white', labelsize=8)
    ax.set_title('XAUUSD — SMC Chart', color='white', fontsize=14)
    ax.grid(False)
    plt.tight_layout()
    plt.show()
    return fig


# สร้างกราฟ!
fig = plot_smc_chart(df, indicators, last_n=50)

## 📝 Cell 5: สร้างฟังก์ชัน get_smc_data() — รวมทุกอย่าง

ฟังก์ชันนี้จะถูกใช้ใน Notebook 03-05 สำหรับส่งข้อมูลให้ LLM

In [ ]:
def get_smc_data(timeframe='15m', count=200):
    """
    ฟังก์ชันรวม: ดึงข้อมูล + คำนวณ indicators → return dict
    ใช้แทน requests.get(backend_url) จาก version เดิม
    """
    df = get_data(timeframe=timeframe, count=count)
    indicators = compute_smc_indicators(df)
    return {'df': df, 'indicators': indicators, 'timeframe': timeframe}


def summarize_smc(data):
    """
    สรุป SMC indicators เป็น text สำหรับส่งให้ LLM
    """
    df = data['df']
    ind = data['indicators']
    tf = data['timeframe']
    last = df.iloc[-1]
    
    lines = [f'=== SMC Analysis ({tf}, {len(df)} candles) ===']
    lines.append(f'ราคาปัจจุบัน: {last["close"]:.2f}')
    lines.append(f'High: {df["high"].max():.2f} | Low: {df["low"].min():.2f}')
    
    # FVG
    fvg = ind['fvg']
    fb = ((fvg['FVG']==1) & fvg['MitigatedIndex'].isna()).sum()
    fbe = ((fvg['FVG']==-1) & fvg['MitigatedIndex'].isna()).sum()
    lines.append(f'\nFVG Active: Bullish={fb}, Bearish={fbe}')
    
    # BOS/CHOCH
    bc = ind['bos_choch']
    bos_mask = bc['BOS'].notna()
    choch_mask = bc['CHOCH'].notna()
    if bos_mask.any():
        last_bos = bc[bos_mask].iloc[-1]
        d = 'Bullish' if last_bos['BOS']==1 else 'Bearish'
        lines.append(f'Last BOS: {d} ที่ {last_bos["Level"]:.2f}')
    if choch_mask.any():
        last_ch = bc[choch_mask].iloc[-1]
        d = 'Bullish' if last_ch['CHOCH']==1 else 'Bearish'
        lines.append(f'Last CHOCH: {d} ที่ {last_ch["Level"]:.2f}')
    
    # OB
    ob = ind['ob']
    obu = ((ob['OB']==1) & ob['MitigatedIndex'].isna()).sum()
    obe = ((ob['OB']==-1) & ob['MitigatedIndex'].isna()).sum()
    lines.append(f'\nOB Active: Bullish={obu}, Bearish={obe}')
    
    # Liquidity
    liq = ind['liquidity']
    bsl = ((liq['Liquidity']==1) & liq['Swept'].isna()).sum()
    ssl = ((liq['Liquidity']==-1) & liq['Swept'].isna()).sum()
    lines.append(f'Liquidity: BSL={bsl}, SSL={ssl}')
    
    return '\n'.join(lines)


# ทดสอบ
data = get_smc_data('15m', 200)
print(summarize_smc(data))

## ✅ สรุป Notebook 01

คุณได้เรียนรู้:
1. ✅ ดึงข้อมูลจาก MT5 โดยตรง (+ CSV fallback)
2. ✅ คำนวณ SMC indicators ด้วย `smartmoneyconcepts` library
3. ✅ สร้างกราฟ candlestick + indicators ด้วย matplotlib
4. ✅ สร้าง `get_smc_data()` และ `summarize_smc()` สำหรับใช้ใน notebook ถัดไป

**ทั้งหมดนี้ไม่ต้องรัน backend เลย!**

---
**➡️ ต่อไป**: `02_smc_indicators_explained.ipynb` — เข้าใจ indicators แต่ละตัว